# Práctica 1 - Pipeline alternativo de Machine Learning

En esta práctica se construye un pipeline alternativo completo de Machine Learning para predecir el impago de préstamos. Para ello, se implementa una nueva clase de preprocesamiento y una nueva clase de filtrado, siguiendo el patrón `fit/transform` para evitar data leakage. Posteriormente, se entrenan tres modelos de distintas familias: un ensemble de árboles, una máquina de soporte vectorial y una red neuronal. Finalmente, se comparan sus resultados con el modelo base visto en clase.

In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve, auc

from src.preprocessing.practica1_preprocessing import Practica1Preprocess
from src.filtering.practica1_filtering import Practica1Filtering

In [99]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="During the encoding, NaN values were introduced*"
)

## 1. Preprocesamiento de los datos

En primer lugar, se aplica la clase `Practica1Preprocess`, que utiliza el fichero `variables_withExperts.xlsx` para incluir también las variables procedentes de evaluaciones de expertos, como `grade`, `sub_grade`, `int_rate`, `installment` y las variables relacionadas con FICO. Esto permite trabajar con un conjunto de información más completo que el utilizado en el pipeline base.

El preprocesamiento sigue el patrón `fit/transform`: todos los objetos que aprenden información de los datos, como imputadores, codificadores y escaladores, se ajustan únicamente sobre el conjunto de entrenamiento y después se aplican tanto a train como a test. De esta forma se evita introducir información del test durante el entrenamiento.

Respecto a la clase base, se han introducido varias técnicas alternativas. Para los valores nulos, las variables numéricas se imputan con la mediana y las categóricas se imputan con una categoría constante `"missing"`, lo que permite conservar la información de que el dato original no estaba disponible. Para las variables categóricas, `grade` y `sub_grade` se tratan con `OrdinalEncoder`, ya que tienen un orden natural de riesgo, mientras que el resto de variables categóricas se codifican mediante frecuencia usando `CountFrequencyEncoder`. Para las variables numéricas se utiliza `RobustScaler`, que es menos sensible a valores extremos que otros escalados clásicos.

Además, se generan nuevas variables con interpretación financiera, como `fico_mean`, que resume el rango FICO, `installment_income_ratio`, que aproxima el peso de la cuota mensual sobre los ingresos, y `loan_income_ratio`, que relaciona el importe del préstamo con los ingresos anuales. Estas variables buscan aportar información más directamente relacionada con la capacidad de pago del cliente.

In [100]:
preprocessor = Practica1Preprocess(
    var_to_process="data/variables_withExperts.xlsx",
    target="loan_status"
)

preprocessor.fit("data/df_train_small.csv")

X_train_pre, y_train = preprocessor.transform("data/df_train_small.csv")
X_test_pre, y_test = preprocessor.transform("data/df_test_small.csv")

preprocessor.print_summary()

print("Shape X_train después de preprocesamiento:", X_train_pre.shape)
print("Shape X_test después de preprocesamiento:", X_test_pre.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)

RESUMEN DEL PREPROCESAMIENTO PRACTICA 1
Variables predictoras usadas: 102
Variables numéricas originales: 92
Variables categóricas originales: 15
Variables ordinales: ['grade', 'sub_grade']
Variables con frequency encoding: 13
Variables numéricas finales escaladas: 107
Shape X_train después de preprocesamiento: (80000, 107)
Shape X_test después de preprocesamiento: (20000, 107)
Shape y_train: (80000,)
Shape y_test: (20000,)


## 2. Filtrado de variables

Una vez preprocesados los datos, se aplica la clase `Practica1Filtering`, que sigue también el patrón `fit/transform`. El filtrado se ajusta únicamente con el conjunto de entrenamiento y después se aplica a train y test, evitando así data leakage.

El filtrado propuesto es alternativo al de la clase base y se compone de dos pasos. En primer lugar, se aplica `VarianceThreshold` para eliminar variables con varianza nula o muy baja, ya que estas variables apenas aportan información discriminante al modelo. En segundo lugar, se utiliza `SelectKBest` con `mutual_info_classif`, que selecciona las variables con mayor relación estadística con la variable objetivo.

En esta práctica se fija `k_best=80`, de forma que el número final de variables queda reducido a 80. Esta decisión busca mantener un equilibrio entre conservar suficiente información predictiva y reducir la dimensionalidad, simplificando el entrenamiento posterior de los modelos.

In [101]:
print("NaN en X_train_pre:", X_train_pre.isna().sum().sum())
print("NaN en X_test_pre:", X_test_pre.isna().sum().sum())

NaN en X_train_pre: 0
NaN en X_test_pre: 0


In [102]:
filtering = Practica1Filtering(variance_threshold=0.0, k_best=80)

filtering.fit(X_train_pre, y_train)

X_train_filt = filtering.transform(X_train_pre)
X_test_filt = filtering.transform(X_test_pre)

filtering.print_summary()

print("Shape X_train después de filtrado:", X_train_filt.shape)
print("Shape X_test después de filtrado:", X_test_filt.shape)

RESUMEN DEL FILTRADO PRACTICA 1
Features iniciales:                    107
Tras filtro de varianza:              107
Features seleccionadas finales:       80
Shape X_train después de filtrado: (80000, 80)
Shape X_test después de filtrado: (20000, 80)


## 3. Entrenamiento de modelos

Una vez finalizado el preprocesamiento y el filtrado, se entrenan tres modelos de aprendizaje automático pertenecientes a familias distintas. En concreto, se utilizará un ensemble de árboles de decisión, una máquina de soporte vectorial y una red neuronal. El objetivo es comparar su rendimiento en la predicción de impago sobre el conjunto de test.

### 3.1 Ensemble de árboles

Como modelo de tipo ensemble se ha elegido un `HistGradientBoostingClassifier`. Este modelo pertenece a la familia de ensembles de árboles y suele funcionar bien en problemas tabulares, ya que combina varios árboles de decisión construidos de forma secuencial para corregir errores del modelo anterior.

Se ha utilizado `max_iter=200` para permitir un número suficiente de iteraciones de boosting, `learning_rate=0.05` para que el aprendizaje sea más gradual y `max_leaf_nodes=31` para limitar la complejidad de cada árbol. Además, se ha aplicado `class_weight={0: 1, 1: 3}` para dar más peso a la clase de impago, ya que se trata de la clase minoritaria y es la más importante desde el punto de vista del problema.

Esta ponderación no es tan agresiva como `class_weight="balanced"`, pero permite mejorar la detección de impagos sin reducir demasiado la accuracy global. Por tanto, se busca un compromiso entre detectar más clientes con riesgo y mantener un número razonable de aciertos totales.

In [103]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    class_weight={0: 1, 1: 3},
    random_state=42
)

hgb_model.fit(X_train_filt, y_train.values.ravel());

### 3.2 Máquina de soporte vectorial

Como representante de las máquinas de soporte vectorial se emplea un modelo `SVC` con kernel radial (`rbf`). Este tipo de modelo puede capturar fronteras de decisión no lineales y resulta útil como comparación frente a los modelos basados en árboles.

Se activa la opción `probability=True` para poder obtener probabilidades estimadas y calcular posteriormente la métrica PR-AUC. Además, se utiliza `class_weight="balanced"` para compensar parcialmente el desbalanceo entre clases. Como las SVM pueden ser computacionalmente costosas con conjuntos de datos grandes, el entrenamiento se realiza sobre una muestra del conjunto de train, manteniendo `random_state=42` para que el resultado sea reproducible.

In [104]:
# Para que no sea excesivamente lento, se toma una muestra del train
X_train_svm = X_train_filt.sample(n=10000, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_svm, y_train_svm.values.ravel());

### 3.3 Red neuronal

Como tercer modelo se entrena una red neuronal multicapa mediante `MLPClassifier`. Este modelo constituye una familia distinta a los ensembles de árboles y a las SVM, y permite aproximar relaciones no lineales complejas entre las variables de entrada y la probabilidad de impago.

Se utiliza una arquitectura sencilla con dos capas ocultas de 64 y 32 neuronas, activación `relu` y optimizador `adam`. También se aplica `early_stopping=True`, de forma que el entrenamiento puede detenerse si el rendimiento deja de mejorar, reduciendo el riesgo de sobreajuste. El objetivo no es realizar una optimización exhaustiva de hiperparámetros, sino comparar una red neuronal razonable frente a los otros modelos solicitados.

In [105]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=400,
    early_stopping=True,
    random_state=42
)

mlp_model.fit(X_train_filt, y_train.values.ravel());

## 4. Evaluación de modelos

Para comparar los modelos entrenados, se calculan las métricas solicitadas en el conjunto de test: `accuracy`, `precision`, `recall` y `PR-AUC`.

La clase positiva se define como impago, es decir, `loan_status != "Fully Paid"`. Esto es importante porque las métricas de precision y recall se interpretan respecto a la clase de interés: los clientes que no devuelven completamente el préstamo.

En este problema, la accuracy debe interpretarse con cuidado, ya que las clases están desbalanceadas. Un modelo podría obtener una accuracy aparentemente alta prediciendo mayoritariamente la clase mayoritaria, pero ser poco útil para detectar impagos. Por eso, además de la accuracy, se presta especial atención a la precision, al recall y a la PR-AUC.

La precision indica qué proporción de los clientes clasificados como impago realmente lo son. El recall indica qué proporción de todos los impagos reales consigue detectar el modelo. La PR-AUC resume el compromiso entre precision y recall para distintos umbrales, siendo especialmente útil en problemas con clase minoritaria.

In [106]:
def evaluate_model(model, X_test, y_test, model_name):
    y_true = y_test.values.ravel()
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_vals, precision_vals)

    results = {
        "Modelo": model_name,
        "Accuracy": accuracy,
        "Precision (impago)": precision,
        "Recall (impago)": recall,
        "PR-AUC": pr_auc
    }

    return results

In [107]:
results_hgb = evaluate_model(hgb_model, X_test_filt, y_test, "HistGradientBoosting")
results_svm = evaluate_model(svm_model, X_test_filt, y_test, "SVM")
results_mlp = evaluate_model(mlp_model, X_test_filt, y_test, "MLP")

In [108]:
# Cálculo del PR-AUC del modelo base visto en clase
# El modelo base usa el FICO score normalizado como aproximación simple al riesgo.

df_base = pd.read_csv("data/df_test_small.csv")

df_fico = (
    df_base.loc[:, ["fico_range_low", "fico_range_high", "loan_status"]]
    .assign(prob_low=lambda x: (x.fico_range_low - 300) / (850 - 300))
    .assign(prob_high=lambda x: (x.fico_range_high - 300) / (850 - 300))
    .assign(prob_paid=lambda x: (x.prob_low + x.prob_high) / 2)
)

y_base_default = (df_fico["loan_status"] != "Fully Paid").astype(int)

# Como prob_paid mide probabilidad aproximada de pagar,
# para la clase positiva de impago usamos 1 - prob_paid.
prob_default = 1 - df_fico["prob_paid"]

precision_vals, recall_vals, _ = precision_recall_curve(
    y_base_default,
    prob_default
)

base_pr_auc = auc(recall_vals, precision_vals)

base_results = {
    "Modelo": "Modelo base",
    "Accuracy": 0.72,
    "Precision (impago)": 0.26,
    "Recall (impago)": 0.24,
    "PR-AUC": base_pr_auc
}

results_df = pd.DataFrame([
    base_results,
    results_hgb,
    results_svm,
    results_mlp
])

results_df

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.7200,0.260000,0.240000,0.291412
1,HistGradientBoosting,0.7147,0.359711,0.548161,0.389932
2,SVM,0.6931,0.228919,0.226170,0.230055
3,MLP,0.7973,0.462351,0.087566,0.307711


In [109]:
results_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision (impago)": "{:.3f}",
    "Recall (impago)": "{:.3f}",
    "PR-AUC": "{:.3f}"
})

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.720,0.260,0.240,0.291
1,HistGradientBoosting,0.715,0.360,0.548,0.390
2,SVM,0.693,0.229,0.226,0.230
3,MLP,0.797,0.462,0.088,0.308


## 5. Comparación con el modelo base

La tabla de resultados permite comparar los tres modelos entrenados con el modelo base de referencia visto en clase. El modelo base obtenía aproximadamente una accuracy de 0.720, una precision de 0.260 y un recall de 0.240 para la clase de impago. Además, en este notebook se ha calculado también su PR-AUC usando el FICO score normalizado como aproximación a la probabilidad de pago y transformándolo a probabilidad de impago mediante `1 - prob_paid`.

El modelo **HistGradientBoosting** es el que ofrece el mejor equilibrio global entre las métricas relevantes. Su accuracy es de 0.715, muy cercana a la del modelo base, aunque ligeramente inferior. Sin embargo, mejora de forma clara la detección de impagos: la precision sube de 0.260 a 0.360 y el recall pasa de 0.240 a 0.548. Esto significa que el modelo identifica una proporción mucho mayor de clientes que realmente acaban en impago. Además, obtiene la mejor PR-AUC de todos los modelos, con un valor de 0.390 frente al 0.291 del modelo base.

La única métrica en la que el HistGradientBoosting no supera al modelo base es la accuracy, aunque la diferencia es muy pequeña. Esto ocurre porque el modelo asigna más peso a la clase minoritaria y, por tanto, predice más casos como posibles impagos. Al hacerlo, detecta más impagos reales, pero también comete más falsos positivos, lo que puede reducir ligeramente la accuracy global. Para intentar mejorar esta situación se podría ajustar el peso de la clase positiva, probar otros valores de `learning_rate`, `max_iter` o `max_leaf_nodes`, o realizar una búsqueda de hiperparámetros con validación cruzada. También podría ajustarse el umbral de decisión usando un conjunto de validación, buscando un equilibrio entre accuracy y recall.

El modelo **SVM** presenta un rendimiento inferior al modelo base en casi todas las métricas relevantes. Su accuracy es de 0.693, su precision es de 0.229 y su recall es de 0.226, por debajo de los valores del baseline. Además, su PR-AUC es de 0.230, también inferior a la del modelo base. Esto indica que, con la configuración utilizada, la SVM no consigue separar correctamente los casos de impago. Una posible mejora sería entrenarla con una muestra mayor, probar otros valores de `C` y `gamma`, utilizar otro kernel o ajustar mejor el escalado y la selección de variables. Sin embargo, aumentar mucho la muestra también incrementaría considerablemente el coste computacional.

Por su parte, la **red neuronal MLP** obtiene la mejor accuracy de todos los modelos, con un valor de 0.797, y también alcanza la mayor precision, con 0.462. Sin embargo, su recall es muy bajo, solo 0.088, lo que significa que detecta muy pocos impagos reales. Este comportamiento sugiere que la red neuronal tiende a predecir mayoritariamente la clase mayoritaria, es decir, préstamos pagados, y por eso consigue muchos aciertos globales pero falla en el objetivo principal del problema. Para mejorarla se podrían probar pesos de clase mediante técnicas de reponderación, modificar el umbral de decisión, ajustar la arquitectura de la red, aumentar la regularización o usar estrategias de validación para seleccionar mejor los hiperparámetros.

En conjunto, los resultados muestran que no basta con mirar la accuracy. Si el objetivo principal fuera acertar el mayor número total de casos, el MLP parecería el mejor modelo. Sin embargo, en un problema de detección de impago es más importante identificar correctamente a los clientes con riesgo. Desde este punto de vista, el HistGradientBoosting es la opción más adecuada, porque mantiene una accuracy cercana al baseline y mejora claramente la precision, el recall y la PR-AUC de la clase de impago.

## 6. Conclusiones

En esta práctica se ha construido un pipeline alternativo completo de Machine Learning para la predicción de impago. El pipeline incluye una clase de preprocesamiento propia, una clase de filtrado alternativa y el entrenamiento de tres modelos pertenecientes a familias distintas: un ensemble de árboles, una máquina de soporte vectorial y una red neuronal.

En el preprocesamiento se han utilizado las variables de expertos incluidas en `variables_withExperts.xlsx`, lo que permite incorporar información financiera y de riesgo adicional respecto al pipeline base. También se han aplicado técnicas alternativas de imputación, codificación y escalado. En concreto, se ha usado una categoría constante para imputar valores categóricos desconocidos, codificación ordinal para variables con orden natural, codificación por frecuencia para el resto de categóricas y `RobustScaler` para las variables numéricas. Además, se han creado nuevas variables con significado financiero, como `fico_mean`, `installment_income_ratio` y `loan_income_ratio`.

Tras el preprocesamiento, el conjunto de datos queda formado por 107 variables. Posteriormente, el filtrado mediante `VarianceThreshold` y `SelectKBest` reduce el conjunto final a 80 variables. Esta reducción permite simplificar el problema y conservar las variables que presentan mayor relación con la variable objetivo según información mutua.

En cuanto a los modelos, el **HistGradientBoosting** es el modelo más adecuado para el objetivo de la práctica. Aunque su accuracy, 0.715, queda ligeramente por debajo de la del modelo base, 0.720, consigue una mejora clara en las métricas más importantes para la detección de impago. La precision aumenta de 0.260 a 0.360, el recall sube de 0.240 a 0.548 y la PR-AUC mejora de 0.291 a 0.390. Esto significa que el modelo detecta muchos más impagos reales y ofrece un mejor comportamiento global sobre la clase minoritaria.

La **SVM** no consigue mejorar al modelo base. Sus valores de accuracy, precision, recall y PR-AUC son inferiores a los del baseline, lo que indica que esta configuración no es adecuada para este problema. Esto podría deberse a la dificultad de la SVM para trabajar con este conjunto de datos, al tamaño limitado de la muestra utilizada para entrenarla o a la necesidad de ajustar mejor hiperparámetros como `C`, `gamma` o el kernel.

El **MLP** consigue la mayor accuracy y la mayor precision, pero presenta un recall muy bajo. Esto significa que, aunque acierta muchos casos en conjunto, detecta muy pocos impagos reales. En un problema bancario, este comportamiento no es deseable si el objetivo principal es identificar clientes con riesgo, ya que dejaría pasar muchos casos de impago sin detectar.

En conclusión, el pipeline propuesto mejora al modelo base en la detección de impagos, especialmente mediante el modelo HistGradientBoosting. La práctica muestra que, en problemas desbalanceados, la accuracy no es suficiente para evaluar correctamente un modelo. Es necesario analizar también precision, recall y PR-AUC, ya que estas métricas permiten valorar mejor el rendimiento sobre la clase minoritaria. Para mejorar aún más los resultados, se podrían ajustar hiperparámetros con validación cruzada, modificar los pesos de clase, probar distintos umbrales de decisión o incorporar nuevas variables derivadas con significado financiero.